## Implement RAG with Azure PostgreSQL

### Installing Libraries and Utilities

In [ ]:
%pip install psycopg[binary]==3.3.4 psycopg_pool==3.3.1 python-dotenv openai==2.38.0

### Setting up the Environment

In [ ]:
import os
from dotenv import load_dotenv


load_dotenv()

# Loading the database configurations
host = os.getenv("DATABASE_HOSTNAME")
db_name = os.getenv("DATABASE_NAME")
username = os.getenv("DATABASE_USERNAME")
password = os.getenv("DATABASE_PASSWORD")

# Loading the Azure OpenAI configurations
azure_openai_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
azure_openai_key = os.getenv("AZURE_OPENAI_KEY")
embedding_model_name = os.getenv("EMBEDDING_MODEL_NAME")
chat_completions_model_name = os.getenv("CHAT_COMPLETIONS_MODEL_NAME")

### Create the Connection Pool

In [ ]:
from psycopg_pool import ConnectionPool

pool = ConnectionPool(
    conninfo=(
        f"host={host} "
        f"dbname={db_name} "
        f"user={username} "
        f"password={password} "
        f"sslmode=require"
    ),
    min_size=2,
    max_size=10
)

pool.wait()

print("Connection pool created successfully")

### Create the Azure OpenAI Client

In [ ]:
from openai import AzureOpenAI

azure_openai_client = AzureOpenAI(
    api_key=azure_openai_key,
    azure_endpoint=azure_openai_endpoint,
    api_version="2024-06-01"
)

### Create the Embedding Generator Helper Function

In [ ]:
def generate_embeddings(text):

    response = azure_openai_client.embeddings.create(
        model=embedding_model_name,
        input=text
    )

    return response.data[0].embedding

### Generate Embeddings for the User Query

Some other questions to ask:
1) What is GreenSteel Ltd target year for achieving carbon neutrality?
2) By how much did FutureEnergy Corp increase its renewable generation capacity?
3) Compare the Scope 3 emissions challenges faced by GreenSteel and UrbanRetail. Also compare their company performance according to their ESG reports.
4) Which company has made the most progress on emission reductions, and why?

In [ ]:
user_query = "Compare the Scope 3 emissions challenges faced by GreenSteel and UrbanRetail. Also compare their company performance according to their ESG reports"

query_embedding = generate_embeddings(user_query)

### Retrieve Context

In [ ]:
search_query = """
SELECT

    ChunkID,
    CompanyName,

    (
        (
            1 -
            (
                ChunkEmbedding <=> %s::vector
            )
        ) * 0.7

        +

        ts_rank(
            to_tsvector(
                'english',
                ChunkText
            ),
            plainto_tsquery(
                'english',
                %s
            )
        ) * 0.3

    ) AS hybrid_score,

    ChunkText

FROM RAG.ESG_Chunks

WHERE

    to_tsvector(
        'english',
        ChunkText
    )

    @@

    plainto_tsquery(
        'english',
        %s
    )

    OR

    (
        ChunkEmbedding <=> %s::vector
    ) < 0.5

ORDER BY hybrid_score DESC

LIMIT 5
"""

In [ ]:
from psycopg.rows import dict_row
import json

with pool.connection() as conn:

    with conn.cursor(
        row_factory=dict_row
    ) as cur:

        cur.execute(
            search_query,
            (
                query_embedding,  # ChunkEmbedding <=> %s::vector
                user_query,    # plainto_tsquery text
                user_query,    # plainto_tsquery text
                query_embedding   # ChunkEmbedding <=> %s::vector
            )
        )

        results = cur.fetchall()

rag_context = {
    "user_query": user_query,
    "documents": [
        {
            "chunk_id": result["chunkid"],
            "company_name": result["companyname"],
            "hybrid_score": float(result["hybrid_score"]),
            "content": result["chunktext"]
        }
        for result in results
    ]
}

print(json.dumps(rag_context, indent=4))

### Setting the System Prompt for LLM Agent

In [ ]:
SYSTEM_PROMPT = """
You are an ESG and Sustainability Reporting Assistant for CarbonOps.

Your purpose is to answer user questions using only the information provided in the retrieved context. The retrieved context consists of ESG reports, sustainability disclosures, reviews, and company-specific ESG information.

Instructions:

1. Use only the information available in the provided context.
2. Do not make assumptions or invent facts.
3. If the answer cannot be determined from the context, clearly state:
   "I could not find sufficient information in the retrieved documents to answer that question."
4. When multiple companies are present in the retrieved context, clearly identify which company each statement refers to.
5. Summarize information in a concise, professional, and business-friendly manner.
6. When discussing ESG topics, pay attention to:
   - Carbon emissions
   - Scope 1, Scope 2, and Scope 3 emissions
   - Energy consumption
   - Renewable energy initiatives
   - Sustainability goals
   - ESG ratings and scores
   - Governance and compliance efforts
   - Risks and opportunities
7. If the user asks for a comparison, compare only the information present in the retrieved context.
8. If numerical values such as emissions, energy consumption, percentages, or ESG scores are present in the context, include them whenever relevant.
9. Do not mention embeddings, vector databases, chunking, retrieval pipelines, similarity scores, vector search, hybrid search, or any implementation details.
10. Maintain a professional consultant-style tone suitable for ESG analysts, sustainability managers, auditors, executives, and business stakeholders.

Response Guidelines:

- Answer the user's question directly.
- Use bullet points when presenting multiple findings.
- Highlight key ESG metrics when available.
- Provide a brief conclusion when appropriate.
- If the retrieved context contains conflicting information, explain the discrepancy.
- At the end of your response include a "Sources Used" section listing the companies referenced in the retrieved context.

The retrieved context provided in the conversation is the authoritative source of truth.
"""

### Sending API Call to LLM

In [ ]:
response = azure_openai_client.chat.completions.create(
    model=chat_completions_model_name,
    messages=[
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": f"""
        Question:
        {user_query}

        Retrieved Context:
        {rag_context}
        """
                }
        ],
        temperature=0.2
)

print(response.choices[0].message.content)